In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc qiskit-ibm-catalog
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# QESEM: Uma Qiskit Function da Qedma
*Veja a [referência da API](https://docs.quantum.ibm.com/api/functions/qedma-qesem)*

> **Note:** As Qiskit Functions são um recurso experimental disponível apenas para usuários dos planos IBM Quantum&reg; Premium, Flex e On-Prem (via IBM Quantum Platform API). Elas estão em status de lançamento em pré-visualização e sujeitas a alterações.

## Visão geral
Embora as unidades de processamento quântico tenham melhorado enormemente nos últimos anos, os erros causados por ruído e imperfeições no hardware existente continuam sendo um desafio central para os desenvolvedores de algoritmos quânticos. À medida que a área se aproxima de computações quânticas em escala utilitária que não podem ser verificadas classicamente, soluções para cancelar ruído com precisão garantida tornam-se cada vez mais importantes. Para superar esse desafio, a Qedma desenvolveu o Quantum Error Mitigation (QESEM), integrado de forma transparente na IBM Quantum Platform como uma [Qiskit Function](/guides/functions).

Com o QESEM, os usuários podem executar seus circuitos quânticos em QPUs ruidosas para obter resultados altamente precisos e livres de erros, com sobrecargas de tempo de QPU altamente eficientes, próximas dos limites fundamentais. Para isso, o QESEM utiliza um conjunto de métodos proprietários desenvolvidos pela Qedma para caracterização e redução de erros. As técnicas de redução de erros incluem otimização de gates, transpilação ciente de ruído, supressão de erros (ES) e mitigação de erros imparcial (EM). Com essa combinação de métodos baseados em caracterização, os usuários podem obter resultados confiáveis e livres de erros para circuitos quânticos genéricos de grande volume, viabilizando aplicações que não poderiam ser realizadas de outra forma.

Para uma descrição completa dos componentes subjacentes, bem como uma demonstração em escala utilitária, consulte o artigo [Reliable high-accuracy error mitigation for utility-scale quantum circuits](https://arxiv.org/abs/2508.10997).
## Descrição
Você pode usar a função QESEM da Qedma para estimar e executar facilmente seus circuitos com supressão e mitigação de erros, alcançando volumes maiores de circuitos e maiores precisões. Para usar o QESEM, você fornece um circuito quântico, um conjunto de observáveis a medir, uma precisão estatística alvo para cada observável e uma QPU escolhida. Antes de executar o circuito com a precisão alvo, você pode estimar o tempo de QPU necessário com base em um cálculo analítico que não requer a execução do circuito. Quando estiver satisfeito com a estimativa de tempo de QPU, você pode executar o circuito com o QESEM.

Quando você executa um circuito, o QESEM roda um protocolo de caracterização do dispositivo adaptado ao seu circuito, gerando um modelo de ruído confiável para os erros que ocorrem no circuito. Com base na caracterização, o QESEM primeiro implementa transpilação ciente de ruído para mapear o circuito de entrada em um conjunto de qubits físicos e gates, minimizando o ruído que afeta o observável alvo. Isso inclui os gates nativamente disponíveis (CX/CZ em dispositivos IBM&reg;), bem como gates adicionais otimizados pelo QESEM, formando o conjunto estendido de gates do QESEM. O QESEM então executa um conjunto de circuitos ES e EM baseados em caracterização na QPU e coleta os resultados das medições. Esses resultados são então pós-processados classicamente para fornecer um valor esperado imparcial e uma barra de erro para cada observável, correspondendo à precisão solicitada.

![Visão geral do Qedma QESEM](../docs/images/guides/qedma-qesem/overview.svg)
O QESEM demonstrou fornecer resultados de alta precisão para uma variedade de aplicações quânticas e nos maiores volumes de circuitos alcançáveis hoje. O QESEM oferece os seguintes recursos voltados ao usuário, demonstrados na seção de benchmarks abaixo:
-	**Precisão garantida:** O QESEM gera estimativas imparciais para os valores esperados dos observáveis. Seu método EM é equipado com garantias teóricas que — em conjunto com a caracterização de ponta da Qedma — asseguram que a mitigação converge para a saída do circuito sem ruído, dentro da precisão especificada pelo usuário. Em contraste com muitos métodos heurísticos de EM propensos a erros sistemáticos ou vieses, a precisão garantida do QESEM é essencial para assegurar resultados confiáveis em circuitos quânticos e observáveis genéricos.
-	**Escalabilidade para grandes QPUs:** O tempo de QPU do QESEM depende dos volumes de circuito, mas é independente do número de qubits. A Qedma demonstrou o QESEM nos maiores dispositivos quânticos disponíveis hoje, incluindo os dispositivos IBM Quantum Eagle de 127 qubits e Heron de 133 qubits.
-	**Independente de aplicação:** O QESEM foi demonstrado em uma variedade de aplicações, incluindo simulação de Hamiltoniano, VQE, QAOA e estimativa de amplitude. Os usuários podem inserir qualquer circuito quântico e observável a ser medido, e obter resultados precisos e livres de erros. As únicas limitações são ditadas pelas especificações de hardware e pelo tempo de QPU alocado, que determinam os volumes de circuito acessíveis e as precisões de saída. Em contraste, muitas soluções de redução de erros são específicas para aplicações ou envolvem heurísticas não controladas, tornando-as inaplicáveis para circuitos e aplicações quânticas genéricas.
-  **Conjunto estendido de gates:** O QESEM suporta gates de ângulo fracionário e fornece gates $Rzz(\theta)$ de ângulo fracionário otimizados pela Qedma nos dispositivos IBM Quantum Heron e Eagle. Esse conjunto estendido de gates permite uma compilação mais eficiente e viabiliza volumes de circuito maiores em até um fator de 2 em comparação com a compilação padrão CX/CZ.
-	**Observáveis multibase:** O QESEM suporta observáveis de entrada compostos por muitas strings de Pauli não comutativas, como Hamiltonianos genéricos. A escolha das bases de medição e a otimização da alocação de recursos de QPU (shots e circuitos) são então realizadas automaticamente pelo QESEM para minimizar o tempo de QPU necessário para a precisão solicitada. Essa otimização, que leva em conta as fidelidades de hardware e as taxas de execução, permite que você execute circuitos mais profundos e obtenha maior precisão.
## Benchmarks
O QESEM foi testado em uma ampla variedade de casos de uso e aplicações. Os exemplos a seguir podem ajudar você a avaliar quais tipos de cargas de trabalho podem ser executadas com o QESEM.

Uma figura de mérito fundamental para quantificar a dificuldade tanto da mitigação de erros quanto da simulação clássica para um dado circuito e observável é o **volume ativo**: o número de gates CNOT que afetam o observável no circuito. O volume ativo depende da profundidade e largura do circuito, do peso do observável e da estrutura do circuito, que determina o cone de luz do observável. Para mais detalhes, veja a palestra do [IBM Quantum Summit 2024](https://www.youtube.com/watch?v=Hd-IGvuARfE&t=1730s). O QESEM oferece valor particularmente alto no regime de alto volume, fornecendo resultados confiáveis para circuitos e observáveis genéricos.

![Volume ativo](../docs/images/guides/qedma-qesem/active_volume.svg)

| Aplicação    | Número de qubits | Dispositivo | Descrição do circuito | Precisão | Tempo total | Uso de runtime |
| ---------  | ---------------- | ----- | -------------------------- | -------- | ---------- | ------------- |
| Circuito VQE  | 8              | Eagle (r3) | 21 camadas no total, 9 bases de medição, cadeia 1D                    | 98%      | 35 min      | 14 min         |
| Kicked Ising   | 28              | Eagle (r3) | 3 camadas únicas x 3 passos, topologia heavy-hex 2D                      | 97%     | 22 min      | 4 min          |
| Kicked Ising   | 28              | Eagle (r3) | 3 camadas únicas x 8 passos, topologia heavy-hex 2D                     | 97%      | 116 min      | 23 min          |
| Simulação de Hamiltoniano Trotterizado   | 40  | Eagle (r3)            | 2 camadas únicas x 10 passos de Trotter, cadeia 1D                    | 97%      | 3 horas     | 25 min         |
| Simulação de Hamiltoniano Trotterizado   | 119  | Eagle (r3)           | 3 camadas únicas x 9 passos de Trotter, topologia heavy-hex 2D                    | 95%      | 6,5 horas     | 45 min         |
| Kicked Ising  | 136             | Heron (r2) | 3 camadas únicas x 15 passos, topologia heavy-hex 2D                    | 99%      | 52 min             | 9 min           |

A precisão é medida aqui em relação ao valor ideal do observável: $\frac{\langle O \rangle_{ideal} - \epsilon}{\langle O \rangle_{ideal}}$, onde '$\epsilon$' é a precisão absoluta da mitigação (definida pelo usuário), e $\langle O \rangle_{ideal}$ é o observável no circuito sem ruído.
'Uso de runtime' mede o uso do benchmark no modo batch (soma do uso de jobs individuais), enquanto 'tempo total' mede o uso no modo session (tempo de parede do experimento), que inclui tempos adicionais de computação clássica e comunicação. O QESEM está disponível para execução em ambos os modos, para que os usuários possam fazer o melhor uso dos seus recursos disponíveis.

Os circuitos Kicked Ising de 28 qubits simulam o Cristal de Quase-Tempo Discreto estudado por Shinjo et al. (veja [arXiv 2403.16718](https://arxiv.org/abs/2403.16718) e [Q2B24 Tokyo](https://www.youtube.com/watch?v=tQW6FdLc6zo)) em três loops conectados do ibm_kawasaki. Os parâmetros de circuito usados aqui são $(\theta_x, \theta_z) = (0.9 \pi, 0)$, com um estado inicial ferromagnético $| \psi_0 \rangle = | 0 \rangle ^{\otimes n}$. O observável medido é o valor absoluto da magnetização $M = |\frac{1}{28} \sum_{i=0}^{27} \langle Z_i \rangle|$. O experimento Kicked Ising em escala utilitária foi executado nos 136 melhores qubits do ibm_fez; este benchmark específico foi executado no ângulo de Clifford $(\theta_x, \theta_z) = (\pi, 0)$, no qual o volume ativo cresce lentamente com a profundidade do circuito, o que — em conjunto com as altas fidelidades do dispositivo — permite alta precisão em um curto tempo de execução.

Os circuitos de simulação de Hamiltoniano Trotterizado são para um modelo de Ising de Campo Transverso em ângulos fracionários: $(\theta_{zz}, \theta_x) = (\pi / 4, \pi /8)$ e $(\theta_{zz}, \theta_x) = (\pi / 6, \pi / 8)$ respectivamente (veja [Q2B24 Tokyo](https://www.youtube.com/watch?v=tQW6FdLc6zo)). O circuito em escala utilitária foi executado nos 119 melhores qubits do ibm_brisbane, enquanto o experimento de 40 qubits foi executado na melhor cadeia disponível. A precisão é reportada para a magnetização; resultados de alta precisão também foram obtidos para observáveis de maior peso.

O circuito VQE foi desenvolvido em conjunto com pesquisadores do Center for Quantum Technology and Applications do Deutsches Elektronen-Synchrotron (DESY). O observável alvo aqui era um Hamiltoniano composto por um grande número de strings de Pauli não comutativas, enfatizando o desempenho otimizado do QESEM para observáveis multibase. A mitigação foi aplicada a um ansatz otimizado classicamente; embora esses resultados ainda não tenham sido publicados, resultados da mesma qualidade serão obtidos para circuitos diferentes com propriedades estruturais semelhantes.
## Primeiros passos
Autentique-se usando sua [chave de API da IBM Quantum Platform](http://quantum.cloud.ibm.com/) e selecione a Qiskit Function QESEM como segue. (Este trecho assume que você já [salvou sua conta](/guides/functions#install-qiskit-functions-catalog-client) no seu ambiente local.)

In [1]:
import qiskit

from qiskit_ibm_catalog import QiskitFunctionsCatalog


catalog = QiskitFunctionsCatalog(channel="ibm_quantum_platform")

# verify that you have access to the function
catalog.list()

[QiskitFunction(qunova/hivqe-chemistry),
 QiskitFunction(global-data-quantum/quantum-portfolio-optimizer),
 QiskitFunction(algorithmiq/tem),
 QiskitFunction(qedma/qesem),
 QiskitFunction(multiverse/singularity),
 QiskitFunction(ibm/circuit-function),
 QiskitFunction(q-ctrl/optimization-solver),
 QiskitFunction(colibritd/quick-pde),
 QiskitFunction(q-ctrl/performance-management),
 QiskitFunction(kipu-quantum/iskay-quantum-optimizer)]

In [2]:
# load the function
qesem_function = catalog.load("qedma/qesem")

## Examples

To get started, try this basic example of estimating the required QPU time to run QESEM for a given `pub`:

In [3]:
# This cell is hidden from users
from qiskit_ibm_runtime import QiskitRuntimeService

service = QiskitRuntimeService()
backend_name = service.least_busy().name

In [4]:
circ = qiskit.QuantumCircuit(5)
circ.cx(0, 1)
circ.cx(2, 3)
circ.cx(1, 2)
circ.cx(3, 4)

avg_magnetization = qiskit.quantum_info.SparsePauliOp.from_sparse_list(
    [("Z", [q], 1 / 5) for q in range(5)], num_qubits=5
)
other_observable = qiskit.quantum_info.SparsePauliOp.from_sparse_list(
    [("ZZ", [0, 1], 1.0), ("XZ", [1, 4], 0.5)], num_qubits=5
)

time_estimation_job = qesem_function.run(
    pubs=[(circ, [avg_magnetization, other_observable])],
    options={
        "estimate_time_only": "analytical",
    },
    backend_name=backend_name,  # E.g. "ibm_fez"
)

time_estimate_result = time_estimation_job.result()

Você pode usar as APIs familiares do Qiskit Serverless para verificar o status da sua carga de trabalho da Qiskit Function ou retornar resultados:

In [5]:
sample_job = qesem_function.run(
    pubs=[(circ, [avg_magnetization, other_observable])],
    backend_name=backend_name,  # E.g. "ibm_fez"
)

O trecho de código a seguir descreve como recuperar a estimativa de tempo de QPU (quando `estimate_time_only` está definido):

In [6]:
# Print the ID so you can use it later, if necessary
print(sample_job.job_id)

print(sample_job.status())
result = sample_job.result()

9a87a23f-82f5-429e-91fb-cc8a9d107980
QUEUED


O trecho de código a seguir demonstra como recuperar os resultados de mitigação (quando `estimate_time_only` não está definido) e as métricas de execução. Eles contêm dados essenciais que permitem uma compreensão mais profunda de como diferentes parâmetros afetam a execução do QESEM. Também pode ser relevante ao escrever um artigo baseado em sua pesquisa.

In [7]:
print(
    f"The estimated QPU time for this PUB is: "
    f"\n{time_estimate_result[0].metadata}"
)

The estimated QPU time for this PUB is: 
{'time_estimation_sec': 1800, 'description': None, 'instance': 'crn:v1:bluemix:public:quantum-computing:us-east:a/6c63dae5281147f1a0449b36e0aaba3a:ae40ab55-8c55-4042-9204-71a6541d56ec::', 'transpilation_level': 'standard', 'parallel_execution': False, 'total_qpu_time': 0, 'empirical_estimation_mitigation_results': None, 'resource_usage': {'RUNNING: MAPPING': {'CPU_TIME': 42.44837867887691, 'GPU_TIME': 0.0, 'QPU_TIME': 0.0}, 'RUNNING: OPTIMIZING_FOR_HARDWARE': {'CPU_TIME': 17.879877626895905, 'GPU_TIME': 0.0, 'QPU_TIME': 0.0}, 'RUNNING: WAITING_FOR_QPU': {'CPU_TIME': 0.0, 'GPU_TIME': 0.0, 'QPU_TIME': 0.0}, 'RUNNING: EXECUTING_QPU': {'CPU_TIME': 0.0, 'GPU_TIME': 0.0, 'QPU_TIME': 0.0}, 'RUNNING: POST_PROCESSING': {'CPU_TIME': 0.0, 'GPU_TIME': 0.0, 'QPU_TIME': 0.0}}}


## Buscar mensagens de erro
Se o status da sua carga de trabalho for ERROR, use `job.result()` para buscar a mensagem de erro da seguinte forma:

In [9]:
results = result[0]
print(f"Mitigated expectation values: \n{results.data.evs}")
print(f"Mitigated error-bar: \n{results.data.stds}")
noisy_results = results.metadata["noisy_results"]
print(f"Noisy expectation values: \n{noisy_results.evs}")
print(f"Noisy error-bar: \n{noisy_results.stds}")
print(f"Total QPU time: \n {results.metadata['total_qpu_time']}")
print(
    f"Gates fidelity measured during the experiment: "
    f"\n {results.metadata['gate_fidelities']}"
)
print(
    f"Total shots / mitigation shots: \n "
    f"{results.metadata['total_shots']} / "
    f"{results.metadata['mitigation_shots']}"
)
print("Transpiled circuits:")
for i, circuit in enumerate(results.metadata["transpiled_circs"]):
    print(f"Circuit {i}:")
    print(f"  Circuit: \n {circuit['circuit']}")
    if "qubit_map" in circuit:
        print(f"  Qubit mapping: \n {circuit['qubit_map']}")
    print(f"  Measurement bases: \n {circuit['num_measurement_bases']}")

Mitigated expectation values: 
[1.00169764 1.00460812]
Mitigated error-bar: 
[0.00155021 0.0099558 ]
Noisy expectation values: 
[0.95717143 0.94271429]
Noisy error-bar: 
[0.00206982 0.00872689]
Total QPU time: 
 150.0
Gates fidelity measured during the experiment: 
 {'CZ': 0.9979051606662408, 'ID1Q': 0.9993865847216725}
Total shots / mitigation shots: 
 495600 / 220400
Transpiled circuits:
Circuit 0:
  Circuit: 
 OPENQASM 3.0;
include "stdgates.inc";
bit[145] c0;
qubit[145] q0;
rz(-pi) q0[143];
rz(0) q0[141];
rz(-pi) q0[140];
sx q0[143];
sx q0[141];
sx q0[140];
rz(-pi/2) q0[143];
rz(pi/2) q0[141];
rz(-pi/2) q0[140];
sx q0[143];
sx q0[141];
sx q0[140];
rz(pi/2) q0[143];
rz(-pi/2) q0[141];
rz(pi/2) q0[140];
barrier q0[140], q0[141], q0[142], q0[143], q0[144];
cz q0[144], q0[143];
cz q0[142], q0[141];
barrier q0[144], q0[143], q0[142], q0[141];
barrier q0[140], q0[141], q0[142], q0[143], q0[144];
rz(-pi) q0[144];
rz(-pi/2) q0[143];
rz(0) q0[142];
rz(-pi/2) q0[141];
sx q0[144];
sx q0[143];

## Fetch error messages

If your workload status is ERROR, use `job.result()` to fetch the error message to fetch the error message as follows:

In [14]:
# Get the result and truncate for readability
result = sample_job.result()
result_str = str(result)
max_length = 500  # Adjust this value as necessary

if len(result_str) > max_length:
    truncated = (
        result_str[:max_length]
        + f"... (truncated {len(result_str) - max_length} characters)"
    )
else:
    truncated = result_str

print(truncated)

PrimitiveResult([PubResult(data=DataBin(evs=np.ndarray(<shape=(2,), dtype=float64>), stds=np.ndarray(<shape=(2,), dtype=float64>), shape=(2,)), metadata={'gate_fidelities': {'CZ': 0.9979051606662408, 'ID1Q': 0.9993865847216725}, 'total_shots': 495600, 'mitigation_shots': 220400, 'transpiled_circs': [{'circuit': 'OPENQASM 3.0;\ninclude "stdgates.inc";\nbit[145] c0;\nqubit[145] q0;\nrz(-pi) q0[143];\nrz(0) q0[141];\nrz(-pi) q0[140];\nsx q0[143];\nsx q0[141];\nsx q0[140];\nrz(-pi/2) q0[143];\nrz(pi... (truncated 4174 characters)


## Obter suporte

A equipe de suporte da Qedma está aqui para ajudar! Se você encontrar algum problema ou tiver dúvidas sobre o uso da Qiskit Function QESEM, não hesite em entrar em contato. Nossa equipe de suporte experiente e atenciosa está pronta para ajudá-lo com qualquer preocupação técnica ou dúvida que você possa ter.

Você pode nos enviar um e-mail para support@qedma.com para obter assistência. Por favor, inclua o máximo de detalhes possível sobre o problema que está enfrentando para nos ajudar a fornecer uma resposta rápida e precisa. Você também pode entrar em contato com seu representante dedicado da Qedma via e-mail ou telefone.

Para nos ajudar a atendê-lo de forma mais eficiente, por favor forneça as seguintes informações ao entrar em contato:

- Uma descrição detalhada do problema
- O ID do job
- Quaisquer mensagens de erro ou códigos relevantes

Estamos comprometidos em fornecer suporte rápido e eficaz para garantir que você tenha a melhor experiência possível com nossa Qiskit Function.

Estamos sempre buscando melhorar nosso produto e agradecemos suas sugestões! Se você tiver ideias sobre como podemos aprimorar nossos serviços ou recursos que gostaria de ver, envie seus pensamentos para support@qedma.com.

## Próximos passos

> **Tip:** - [Solicitar acesso ao Qedma QESEM](https://quantum.cloud.ibm.com/functions?id=qedma-qesem).
> - Visitar a [referência da API](https://docs.quantum.ibm.com/api/functions/qedma-qesem) para esta Qiskit Function.
> - Revisar [Bauman, N. P., et al. (2025). Coupled Cluster Downfolding Theory in Simulations of Chemical Systems on Quantum Hardware. arXiv preprint arXiv:2507.01199](https://arxiv.org/pdf/2507.01199).
> - Revisar [Aharonov, D., et al. (2025). Reliable high-accuracy error mitigation for utility-scale quantum circuits. arXiv preprint arXiv:2508.10997](https://arxiv.org/pdf/2508.10997).